In [1]:
import pandas as pd

df = pd.read_csv('auditory_check/auditory_check.csv')

# Exclude single-syllable words (length == 1, e.g. cue characters).
df_words = df[df['word_text'].str.len() > 1].copy()

# Normalize the result column and count y/n per (design, word_text).
df_words['result'] = df_words['result'].astype(str).str.strip().str.lower()

counts = (
    df_words[df_words['result'].isin(['y', 'n'])]
    .groupby(['design', 'word_text'])['result']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=['y', 'n'], fill_value=0)
    .reset_index()
)

# Rule: remove the word only if it does NOT have strictly more 'y' than 'n'
# (i.e. keep words where n_y > n_n; flag for removal when n_n >= n_y).
df_n = (
    counts[counts['n'] >= counts['y']]
    [['design', 'word_text', 'y', 'n']]
    .sort_values(['design', 'word_text'])
    .reset_index(drop=True)
)

total_words    = counts['word_text'].nunique()
words_to_remove = len(df_n)
percentage     = (words_to_remove / total_words * 100) if total_words > 0 else 0

print(df_n.shape)
print(f"Words flagged for removal (n >= y): {words_to_remove} / {total_words} "
      f"({percentage:.2f}%)")
df_n.head(100)


(11, 4)
Words flagged for removal (n >= y): 11 / 292 (3.77%)


result,design,word_text,y,n
0,four_3_syllable_words,手榴弹,2,4
1,four_3_syllable_words,荧光屏,2,6
2,four_3_syllable_words,进行曲,5,7
3,three_3_syllable_words,奴隶制,5,13
4,three_3_syllable_words,手榴弹,0,17
5,three_3_syllable_words,灵敏度,3,13
6,three_3_syllable_words,畜牧业,4,12
7,three_3_syllable_words,荧光屏,1,17
8,three_3_syllable_words,葡萄糖,4,14
9,three_3_syllable_words,进行曲,8,9


In [2]:
import os

# Get the directory containing the word database files
word_db_dir = 'word_database_txt_v2'

# For each word in df_n, remove it from the corresponding txt file
for _, row in df_n.iterrows():
    design = row['design']
    word_text = row['word_text']
    
    # Construct the file path
    file_path = os.path.join(word_db_dir, f"{design}.txt")
    
    # Check if file exists
    if os.path.exists(file_path):
        # Read all lines from the file
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        # Filter out the word to delete
        filtered_lines = [line for line in lines if line.strip() != word_text]
        
        # Write back the filtered lines
        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(filtered_lines)
        
        print(f"Removed '{word_text}' from {file_path}")
    else:
        print(f"File not found: {file_path}")

print("\nDeletion complete!")

Removed '手榴弹' from word_database_txt_v2/four_3_syllable_words.txt
Removed '荧光屏' from word_database_txt_v2/four_3_syllable_words.txt
Removed '进行曲' from word_database_txt_v2/four_3_syllable_words.txt
Removed '奴隶制' from word_database_txt_v2/three_3_syllable_words.txt
Removed '手榴弹' from word_database_txt_v2/three_3_syllable_words.txt
Removed '灵敏度' from word_database_txt_v2/three_3_syllable_words.txt
Removed '畜牧业' from word_database_txt_v2/three_3_syllable_words.txt
Removed '荧光屏' from word_database_txt_v2/three_3_syllable_words.txt
Removed '葡萄糖' from word_database_txt_v2/three_3_syllable_words.txt
Removed '进行曲' from word_database_txt_v2/three_3_syllable_words.txt
Removed '功败垂成' from word_database_txt_v2/three_4_syllable_words.txt

Deletion complete!
